In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve()
sys.path.append(str(ROOT / "src"))

print("Project root:", ROOT)

In [ ]:
import cv2
import easyocr
import re

image_path = "input3.jpg"

image = cv2.imread(image_path)

if image is None:
    raise FileNotFoundError(f"Could not find image: {image_path}")

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

reader = easyocr.Reader(["en"])
results = reader.readtext(blurred)

def clean_text(text):
    text = re.sub(r"\$?\d+(\.\d+)?", "", text)
    text = re.sub(r"[^a-zA-Z& ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

JUNK_WORDS = [
    "DELICIOUS FOOD",
    "MENU",
    "APPETIZER",
    "APPETIZERS",
    "SALAD & SOUP",
    "SALAD SOUP",
    "DRINKS",
    "D RINKS",
    "FOOD",
]

foodItems = []

for _, text, conf in results:
    if conf > 0.5:
        cleaned = clean_text(text)

        if len(cleaned) > 2 and cleaned.upper() not in JUNK_WORDS:
            foodItems.append(cleaned)

print("Detected food items:")
for item in foodItems:
    print("-", item)

In [ ]:
from predict_allergens import predict_allergens

results = predict_allergens(foodItems)

print("--- Allergen Detection Results ---\n")

for r in results:
    print(r["food_item"])
    print("  Allergens:", ", ".join(r["predicted_allergens"]))
    print("  Matched:", ", ".join(r["matched_ingredients"]))
    print("  Method:", r["method"])
    print("  Confidence:", f"{r['confidence']:.1%}")
    print()

In [ ]:
import pandas as pd

rows = []

for r in results:
    rows.append({
        "food_item": r["food_item"],
        "allergens": ", ".join(r["predicted_allergens"]),
        "matched_ingredients": ", ".join(r["matched_ingredients"]),
        "method": r["method"],
        "confidence": r["confidence"]
    })

df = pd.DataFrame(rows)

output_path = "allergen_results.csv"
df.to_csv(output_path, index=False)

print(f"Saved results to {output_path}")
df.head()